In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
import scienceplots

import time
import math
from bisect import bisect
import os
import sys
import random
from functools import partial
from decimal import Decimal
import numpy as np
import scipy.io as sio
import pysindy as ps
from tqdm import trange

sys.path.insert(0, '../')
from utils import *
from solvel0 import solvel0, MIOSR
from best_subset import backward_refinement, brute_force_all_subsets, brute_force
from UBIC import *
from bayesian_model_evidence import log_evidence

from skimage.restoration import estimate_sigma
import bm3d
from kneed import KneeLocator

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

from rdata import read_rds
from selective_inference import forward_stop_rule

from sklearn.preprocessing import StandardScaler
from sklearn import covariance
from sklearn.linear_model import (LinearRegression, Ridge, BayesianRidge, 
                                    Lasso, lars_path, ElasticNet, MultiTaskLasso, MultiTaskElasticNet, ARDRegression)
from abess import LinearRegression as AbessLinearRegression
from knockpy import KnockoffFilter, knockoff_stats, knockoffs
from knockpy.utilities import estimate_covariance
from scipy import stats
from statsmodels.stats.multitest import multipletests
from c2st.check import c2st # https://github.com/psteinb/c2st

from mbic import mbic, mbic2, ebic

from selective_inference import sfs_si, stepwise_selective_inference, subset_fdr
import fpsample
from dppy.finite_dpps import FiniteDPP

from si4pipeline import (
                        construct_pipelines, 
                        extract_features, 
                        initialize_dataset, 
                        intersection, 
                        lasso, 
                        marginal_screening, 
                        stepwise_feature_selection, 
                        union, 
                        PipelineManager
                        )

from pymcdm import weights as obj_w
from compromise_programming import optimal_decision, compromise_programming, mcdm
from pyRankMCDA.algorithm import rank_aggregation

mrmr is not installed in the env you are using. This may cause an error in future if you try to use the (missing) lib.
L0BnB is not installed.


In [2]:
target_name = 'u'
X_pre = np.load("../Cache/X_pre_GS_2025.npy")
uv_pre = np.load("../Cache/y_pre_GS_2025.npy")

### Ground truth ###
ground_indices_u = (0, 1, 8, 12, 18, 26)
ground_coeff_u = np.array([0.014, -0.014, -1.000, 0.020, 0.020, 0.020])
ground_indices_v = (2, 8, 13, 19, 27)
ground_coeff_v = np.array([-0.067, 1.0, 0.01, 0.01, 0.01])
feature_names = np.load("../Cache/feature_names_GS_2025.npy", allow_pickle=True)

X_pre_top = StandardScaler(with_std=True).fit_transform(X_pre)
uv_pre = StandardScaler(with_std=False).fit_transform(uv_pre)
if target_name == 'u':
    y_pre = uv_pre[:, 0:1]
elif target_name == 'v':
    y_pre = uv_pre[:, 1:2]
else:
    raise ValueError("target_name is either 'u' or 'v'.")